In [115]:
import pandas as pd
import os
import geopandas as gpd

## Collect Linking Tool Results

* **Define the Linking Tool Results Directory:** Assuming you have the following standard file structure:

    ```
    repository
        |___Bidirectional_Linking_Tool
            |___results
        |___BC-CLEWS-Model
            |__from_linking_tool
                |__results
    ```

    - `repository`: All Git Repositories' Root directory.
    - `Bidirectional_Linking_Tool`: Directory containing the _Bidirectional Linking Tool_.
        - `results`: Subdirectory within *Bidirectional_Linking_Tool*.
    - `BC-CLEWS-Model`: Directory containing the _BC-CLEWS-Model_ git repo.
        - `from_linking_tool`: Subdirectory within _BC-CLEWS-Model_.
            - `results`: Subdirectory within *from_linking_tool* , contains the pickle files from Linking tool.

In [116]:
linking_tool_directs={
    'root': 'Bidirectional_Linking_Tool',
    'results': 'results',
}

bc_nexus_direct={
    'root': 'BC-CLEWS-Model',
    'collect_linking_tool_results':'from_linking_tool',
    'results': 'results',
}

print(f"Please make sure you have '{linking_tool_directs['root']}' directory present in your 'repository' root.")

Please make sure you have 'Bidirectional_Linking_Tool' directory present in your 'repository' root.


* Bash Cmd

__!!! Caution__: The Bash cmd execution relies on the aforemention __standard Folder Structure__

> copy-paste result pickle files (__cluster info__ & __timeseries of clusters__) from Linking tool to BC-Nexus directories for further analysis.

In [117]:
!cd .. && cd .. && mkdir -p {bc_nexus_direct['root']}/{bc_nexus_direct['collect_linking_tool_results']}/{bc_nexus_direct['results']} && cp {linking_tool_directs['root']}/{linking_tool_directs['results']}/*.pkl {bc_nexus_direct['root']}/{bc_nexus_direct['collect_linking_tool_results']}/{bc_nexus_direct['results']}

## Define Timeslice conversion results Directory

In [118]:
directories = 'new_sites_timeslice','new_sites'

for directory in directories:
    # Check if the directory exists, and create it if it doesn't
    if not os.path.exists(directory):
        os.makedirs(directory)
        print(f"Directory '{directory}' created successfully.")
    else:
        print(f"Directory '{directory}' already exists.")

Directory 'new_sites_timeslice' already exists.
Directory 'new_sites' already exists.


# Define Season and Timeslice Configuration

> To be replaced by __TSA (Time Slice Aggregation)__ optimization

In [119]:
season_config = {
    "Winter1": {"start": "2021-01-01", "end": "2021-03-19", "daytime": {"day": 8, "night": 17}},
    "Spring": {"start": "2021-03-20", "end": "2021-06-19", "daytime": {"day": 6, "night": 20}},
    "Summer": {"start": "2021-06-20", "end": "2021-09-21", "daytime": {"day": 6, "night": 21}},
    "Fall": {"start": "2021-09-22", "end": "2021-12-20", "daytime": {"day": 8, "night": 18}},
    "Winter2": {"start": "2021-12-21", "end": "2021-12-31", "daytime": {"day": 8, "night": 17}}
}

# Mapping dictionary
timeslice_name_mapping = {
    'Fall_D': 'SEA2D',
    'Fall_N': 'SEA2N',
    'Out_of_Season': 'NA',
    'Spring_D': 'SEA3D',
    'Spring_N': 'SEA3N',
    'Summer_D': 'SEA4D',
    'Summer_N': 'SEA4N',
    'Winter1_D': 'SEA1D',
    'Winter1_N': 'SEA1N',
    'Winter2_D': 'SEA1D',
    'Winter2_N': 'SEA1N'
}

# Define Funcs

> to be automated further inline with timeslice configuration for __BC-Nexus__

In [120]:
def create_timeslice_from_cluster(cluster_ts_pickle,site_name,season_config,timeslice_name_mapping):

    df_ts=pd.read_pickle(cluster_ts_pickle)

    # Convert season_config dates to datetime with error handling
    for season, dates in season_config.items():
        try:
            dates["start"] = pd.to_datetime(dates["start"])
            dates["end"] = pd.to_datetime(dates["end"])
        except ValueError as e:
            print(f"Error converting dates for season '{season}': {e}")

    # Improved function for clarity and error handling
    def get_season_time_of_day(datetime):
        for season, dates in season_config.items():
            if dates["start"] <= datetime <= dates["end"]:
                # Handle specific time ranges for day and night
                if 0 <= datetime.hour <= dates["daytime"]["day"]:
                    return f"{season}_N"  # Night
                elif dates["daytime"]["day"] <= datetime.hour <= dates["daytime"]["night"]:
                    return f"{season}_D"  # Day
                else:  # Outside daytime range
                    return f"{season}_N"  # Assuming night for values outside day/night times
        # Handle date outside any season (optional: return specific value or raise an error)
        return "Out_of_Season"  # Example handling

    # Create the 'season_daytime' column with error handling
    try:
        df_ts['season_daytime'] = df_ts.index.map(get_season_time_of_day)
    except TypeError as e:
        print("Error applying function: ", e)

    # Display the DataFrame
    # Group by season and day/night
    grouped_df = df_ts.groupby(['season_daytime']).agg({
        site_name: ['mean', 'std']  # You can add more aggregation functions here
    })

    # Display the grouped DataFrame
    grouped_df['CF_mean'] = grouped_df[site_name]['mean']
    grouped_df['timeslices'] = grouped_df.index.map(timeslice_name_mapping)
    grouped_df=grouped_df.groupby('timeslices').mean() 

    return grouped_df,df_ts

In [121]:
def create_n_save_timeslice_files(ts_pickle,cluster_pickle,asset_type,season_config,timeslice_name_mapping):
    sites_df=gpd.GeoDataFrame(pd.read_pickle(cluster_pickle))
    sites_without_geom=sites_df.drop(columns=['geometry'])
    sites_without_geom.to_csv(f'new_sites/{asset_type}.csv')
    
    sites=list(sites_df.index)
    cluster_ts_timeslice = {}
    cluster_ts={}

    for site in sites:
        cluster_ts_timeslice[site],cluster_ts = create_timeslice_from_cluster(ts_pickle, site,season_config,timeslice_name_mapping)
        cluster_ts_timeslice[site].to_csv(f'new_sites_timeslice/{asset_type}_{site}.csv')
        
    
    print(f"BCNexus compatible timeseries created for - {asset_type} Clusters")
    return cluster_ts,cluster_ts_timeslice,sites_df

# Generate Results

In [122]:
solar_ts,solar_timeslices,solar_sites=create_n_save_timeslice_files('results/Solar_Top_Sites_Clustered_CF_timeseries.pkl','results/Solar_Top_Sites_Clustered.pkl','Solar',season_config,timeslice_name_mapping)
wind_ts,wind_timeslices,wind_sites=create_n_save_timeslice_files('results/Wind_Top_Sites_Clustered_CF_timeseries.pkl','results/Wind_Top_Sites_Clustered.pkl','Wind',season_config,timeslice_name_mapping)

BCNexus compatible timeseries created for - Solar Clusters
BCNexus compatible timeseries created for - Wind Clusters


# Whiteboard Playground

* Review the Input and Result Files

In [123]:
wind_sites

,Region,Region_ID,Cluster_No,capex,potential_capacity,CF_mean,p_lcoe,geometry,Site_ID
cluster_id,,,,,,,,,
East Kootenay_1,East Kootenay,16,1,1.3234,460.230391,0.489829,3.232099,"MULTIPOLYGON (((-116.62343 50.66577, -116.6234...",1
Kitimat-Stikine_1,Kitimat-Stikine,4,1,1.3234,1965.303155,0.531154,3.217799,"MULTIPOLYGON (((-131.34842 56.62827, -131.3459...",2
Stikine_1,Stikine,2,1,1.3234,1860.175576,0.514496,3.213850,"MULTIPOLYGON (((-130.46843 57.92077, -130.4684...",3
Fraser-Fort George_1,Fraser-Fort George,6,1,1.3234,714.290878,0.544845,3.206377,"MULTIPOLYGON (((-123.96843 54.40327, -123.9709...",4


In [124]:
solar_sites

,Region,Region_ID,Cluster_No,capex,potential_capacity,CF_mean,p_lcoe,geometry,Site_ID
cluster_id,,,,,,,,,
East Kootenay_1,East Kootenay,16,1,1.161269,1000.0,0.244803,1.814032,"MULTIPOLYGON (((-116.27557 49.86490, -116.2755...",1
